# Import libraries and set up paths

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import json
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go

import sys
sys.path.append('../../')
os.chdir('/Users/vanessa/PhD/Dev/SELAnalysis/TAVERN/')

from tavern.orbit_tracks_utils import *
from tavern.config import config as cfg

In [7]:
file_label = '2023-01-01_2024-12-31'
SATs = [file.replace('.parquet','') for file in os.listdir(cfg.data_path) if file.endswith('.parquet')]
POD_CONFIG_PATH = f"{cfg.general_settings['paths']['base']}/configs/POD/config.json"
config = json.load(open(POD_CONFIG_PATH, 'r'))

# Utils

In [3]:
def iqr(x):
    return x.quantile(0.995) - x.quantile(0.005)
def quantile(x):
    return x.quantile(0.995)

def compute_montly_stats(df):
    for i, month in enumerate(df.index.to_period('M').unique().astype(str)):
        df_month = df[df.index.to_period('M').astype(str) == month]
        df_stats = df_month[['orbital_decay_rate']].agg(['min', 'max', 'mean', 'median', 'std', 'kurtosis', 'skew', iqr, quantile])
        df_stats['Function'] = df_stats.index.values
        df_stats['Function'] = df_stats['Function'].apply(lambda x: 'IQR 0.995 - 0.005' if 'iqr' == x else '0.995th percentile' if 'quantile' == x else x)
        df_stats.index = df_stats.reset_index()
        df_stats['Month'] = month
        if i == 0:
            df_all_stats = df_stats
        else:
            df_all_stats = pd.concat([df_all_stats, df_stats])

    df_all_stats['color'] = df_all_stats['Month'].apply(lambda x: f"#{np.random.randint(200, 255):02X}{np.random.randint(200, 255):02X}{np.random.randint(200, 255):02X}")
    df_all_stats.reset_index(drop=True, inplace=True)

    return df_all_stats

def plot_monthly_stats_table(df_all_stats, satellite):
    fig = go.Figure(data=[go.Table(
        header=dict(values=['Month', 'Function', 'Orbital Decay (m)',
        ],
                    fill_color='lavender',
                    align='center'),
        cells=dict(values=[ df_all_stats['Month'], df_all_stats['Function'], df_all_stats['orbital_decay_rate']],
                fill_color=[df_all_stats['color']],
                align='center', format=[None, None, '.2f', '.2f', '.2f', '.0f']))
        ], layout=go.Layout(title=f'Statistics for Orbital Decay for {satellite}'))

    fig.data[0]['columnwidth'] = [70,60,80,100,80]
    fig.update_layout(
        margin=dict(l=20, r=20, t=50, b=10),
        autosize=True, title_x=0.5)
    fig.write_image(f"./data/stats_table_{satellite}_{file_label}.png")

def plot_monthly_stats_bar(df_all_stats, satellite):
    df_all_stats_bar = df_all_stats.pivot(index='Month', columns='Function', values='orbital_decay_rate')
    df_all_stats_bar = df_all_stats_bar.reset_index()
    df_all_stats_bar = df_all_stats_bar.sort_values(by='Month')
    df_all_stats_bar = df_all_stats_bar.set_index('Month')

    fig = make_subplots(rows=3, cols=3, shared_xaxes=True, vertical_spacing=0.02)
    fig.add_trace(go.Bar(name='Min', x=df_all_stats_bar.index, y=df_all_stats_bar['min']), row=1, col=1)
    fig.add_trace(go.Bar(name='Max', x=df_all_stats_bar.index, y=df_all_stats_bar['max']), row=1, col=2)
    fig.add_trace(go.Bar(name='Mean', x=df_all_stats_bar.index, y=df_all_stats_bar['mean']), row=1, col=3)
    fig.add_trace(go.Bar(name='Median', x=df_all_stats_bar.index, y=df_all_stats_bar['median']), row=2, col=1)
    fig.add_trace(go.Bar(name='Standard Deviation', x=df_all_stats_bar.index, y=df_all_stats_bar['std']), row=2, col=2)
    fig.add_trace(go.Bar(name='IQR 0.995-0.005', x=df_all_stats_bar.index, y=df_all_stats_bar['IQR 0.995 - 0.005']), row=2, col=3)
    fig.add_trace(go.Bar(name='0.995th Percentile', x=df_all_stats_bar.index, y=df_all_stats_bar['0.995th percentile']), row=3, col=1)
    fig.add_trace(go.Bar(name='Kurtosis', x=df_all_stats_bar.index, y=df_all_stats_bar['kurtosis']), row=3, col=2)
    fig.add_trace(go.Bar(name='Skewness', x=df_all_stats_bar.index, y=df_all_stats_bar['skew']), row=3, col=3)

    fig.update_xaxes(tickvals=df_all_stats_bar.index)
    fig.update_layout(barmode='group', title=f"Orbital Decay Statistics for {config[satellite]['name'].upper()} in {file_label}", height=900, width=1200)
    fig.write_html(f"./data/stats_barplot_{satellite}_{file_label}.html")

def plot_monthly_boxplot(df_all_stats, satellite):
    fig = go.Figure()
    for i, month in enumerate(df_all_stats['Month'].unique()):
        df_month = df_all_stats[df_all_stats['Month'] == month]
        fig.add_trace(go.Box(y=df_month['orbital_decay_rate'], name=month))

    fig.update_layout(title=f"Orbital Decay Boxplot for {config[satellite]['name'].upper()} in {file_label}", height=900, width=1200)
    fig.write_html(f"./data/stats_boxplot_{satellite}_{file_label}.html")

# Data read, preprocessing and statistics computation

In [8]:
stats_sat = { }
dfs = {}
ssp_computed = False
for i, sat in enumerate(SATs):
    file_path = os.path.join(cfg.data_path, f'{sat}.parquet')
    print(f"Processing {file_path}")
    df = pd.read_parquet(file_path, columns=['aDot_m_s', 'F10.7', 'Hp30', 'SymH', 'lon_ell [deg]', 'lat_ell [deg]'])
    df['orbital_decay_rate'] = - df['aDot_m_s']
    df.index = pd.to_datetime(df.index).tz_convert(None)
    # df = df.resample('5min').mean()
    dfs[sat] = df
    assert df.index.is_monotonic_increasing, f"Index is not sorted for satellite {sat}"

    if not ssp_computed:
        ssp = compute_subsolar_point(df)
        ssp.index = pd.to_datetime(ssp.index)
        ssp_computed = True

    df = df.join(ssp, how='left')
    ssp_dst = compute_subsolar_point_distance(df)
    dfs[sat] = df.join(ssp_dst, how='left')

    satellite = sat.split('.')[0]

    stats_sat[satellite] = {}
    stats_sat[satellite]['median'] = df['orbital_decay_rate'].median()
    stats_sat[satellite]['mean'] = df['orbital_decay_rate'].mean()
    stats_sat[satellite]['max'] = df['orbital_decay_rate'].max()
    stats_sat[satellite]['min'] = df['orbital_decay_rate'].min()

    for m, month in [(5, 'May'), (10, 'October')]:
        stats_sat[satellite][f'median_{month}_2024'] = df[(df.index.month == m) & (df.index.year == 2024)
        ]['orbital_decay_rate'].median()
        stats_sat[satellite][f'mean_{month}_2024'] = df[(df.index.month == m) & (df.index.year == 2024) ]['orbital_decay_rate'].mean()
        stats_sat[satellite][f'max_{month}_2024'] = df[(df.index.month == m) & (df.index.year == 2024) ]['orbital_decay_rate'].max()
        stats_sat[satellite][f'min_{month}_2024'] = df[(df.index.month == m) & (df.index.year == 2024) ]['orbital_decay_rate'].min()

    # df_stats = compute_montly_stats(df)
    # plot_monthly_stats_bar(df_stats, satellite)
    # plot_monthly_boxplot(df_stats, satellite)
    # plot_monthly_stats_table(df_stats, satellite)
    # print(f"Generated plots for satellite: {config[satellite]['name'].upper()}")

Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/SWMB_RDBWFI.parquet


[#################################] 100% de421.bsp


Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/S6A_RDAGFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/S3B_RDAIFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/GFOC_RDCDFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/SWMC_RDCWFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/S3A_RDAFFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/S1A_RDANFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/SWMA_RDAWFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/GFOD_RDDDFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/S2A_RDAZAFI.parquet
Processing ../SpaceWeatherImpact/data/2023-01-01_2024-12-31/processed/S2B_RDAGFI.parquet


In [5]:
for sat in stats_sat.keys():
    stats_sat[sat]['max_to_median_ratio'] = stats_sat[sat]['max'] / stats_sat[sat]['median'] if stats_sat[sat]['median'] != 0 else np.nan
    stats_sat[sat]['mean_to_median_ratio'] = stats_sat[sat]['mean'] / stats_sat[sat]['median'] if stats_sat[sat]['median'] != 0 else np.nan

    for m, month in [(5, 'May'), (10, 'October')]:
        stats_sat[sat][f'max_to_median_ratio_{month}_2024'] = stats_sat[sat][f'max_{month}_2024'] / stats_sat[sat][f'median_{month}_2024'] if stats_sat[sat][f'median_{month}_2024'] != 0 else np.nan
        stats_sat[sat][f'mean_to_median_ratio_{month}_2024'] = stats_sat[sat][f'mean_{month}_2024'] / stats_sat[sat][f'median_{month}_2024'] if stats_sat[sat][f'median_{month}_2024'] != 0 else np.nan

    stats_sat[sat]['max_to_median_ratio_all_but_may_october'] = np.nan
    stats_sat[sat]['mean_to_median_ratio_all_but_may_october'] = np.nan
    median_values = []
    max_values = []
    mean_values = []
    for year in [2023, 2024]:
        for month in range(1, 13):
            if year == 2024 and month in [5, 10]:
                continue
            df_month = df[(df.index.month == month) & (df.index.year == year)]
            if not df_month.empty:
                median_values.append(df_month['orbital_decay_rate'].median())
                max_values.append(df_month['orbital_decay_rate'].max())
                mean_values.append(df_month['orbital_decay_rate'].mean())

    overall_median = np.median(median_values) if median_values else np.nan
    overall_max = np.max(max_values) if max_values else np.nan
    overall_mean = np.mean(mean_values) if mean_values else np.nan
    stats_sat[sat]['max_to_median_ratio_all_but_may_october'] = overall_max / overall_median if overall_median != 0 else np.nan
    stats_sat[sat]['mean_to_median_ratio_all_but_may_october'] = overall_mean / overall_median if overall_median != 0 else np.nan

# Max to median ratios

In [ ]:
df_stats_all = pd.DataFrame(stats_sat).T
df_stats_all = df_stats_all.sort_values(by='max_to_median_ratio', ascending=False)

In [ ]:
sns.set_context("paper")
sns.set_palette("colorblind")
sns.set()
# plt.style.use('dark_background')
plt.rcParams['grid.color'] = "0.1"

plt.rcParams.update({
    'font.size': 24,  #
    'font.family': 'Times New Roman',
    'axes.labelsize': 'large',
    'axes.titlesize': 'x-large',
    'xtick.labelsize': 'medium',
    'ytick.labelsize': 'medium',
    'figure.titlesize': 'xx-large',
    'legend.fontsize': 'medium',
    'axes.titlesize': 'large',
})
df_stats_all_may = df_stats_all.sort_values(by='max_to_median_ratio_May_2024', ascending=False)
df_stats_all_oct = df_stats_all.sort_values(by='max_to_median_ratio_October_2024', ascending=False)
fig = go.Figure()

altitudes = [config[sat]['altitude'] for sat in df_stats_all.index]
# add secondary y-axis for altitude
fig.add_trace(go.Scatter(name='Altitude (km)', x=df_stats_all.index, y=altitudes, mode='lines+markers', yaxis='y2'))

fig.add_trace(go.Bar(name='Max to Median Ratio Overall', x=df_stats_all.index, y=df_stats_all['max_to_median_ratio']))
fig.add_trace(go.Bar(name='Max to Median Ratio May 2024', x=df_stats_all_may.index, y=df_stats_all_may['max_to_median_ratio_May_2024']))
fig.add_trace(go.Bar(name='Max to Median Ratio October 2024', x=df_stats_all_oct.index, y=df_stats_all_oct['max_to_median_ratio_October_2024']))
fig.update_layout(yaxis2=dict(title='Altitude (km)', overlaying='y', side='right', range=[0, 2000]))
# set ylim 0
fig.update_yaxes(range=[0, 80])
# add general max to median ratio
fig.update_layout(barmode='group', title=f"Max to Median Ratio of Orbital Decay for all Satellites in May and October 2024", height=600, width=1000)
fig.show()

In [ ]:
sns.set_context("paper")
sns.set_palette("colorblind")
sns.set()
# plt.style.use('dark_background')
plt.rcParams['grid.color'] = "0.1"

plt.rcParams.update({
    'font.size': 24,  #
    'font.family': 'Times New Roman',
    'axes.labelsize': 'large',
    'axes.titlesize': 'x-large',
    'xtick.labelsize': 'medium',
    'ytick.labelsize': 'medium',
    'figure.titlesize': 'xx-large',
    'legend.fontsize': 'medium',
    'axes.titlesize': 'large',
# hide grid
    'axes.grid': False,
})

altitudes = [config[sat]['altitude'] for sat in df_stats_all.index]

fig, ax1 = plt.subplots(figsize=(18, 8))

plot_data = pd.DataFrame({
    'Satellite': np.tile(df_stats_all.index, 4),
    'Altitude': np.concatenate([altitudes, altitudes, altitudes, altitudes]),
    'Max to Median Ratio': np.concatenate([ df_stats_all['max_to_median_ratio'], df_stats_all['max_to_median_ratio_May_2024'], df_stats_all['max_to_median_ratio_October_2024'], df_stats_all['mean_to_median_ratio_all_but_may_october']]),
    'Category': ['Overall (2023 and 2024)'] * len(df_stats_all.index) +
                ['May 2024'] * len(df_stats_all.index) +
    ['October 2024'] * len(df_stats_all.index) +
    ['Overall (2023 and 2024) excluding May, October 2024'] * len(df_stats_all.index)
})

plot_data.drop(index=plot_data[(plot_data['Category'] == 'Overall (2023 and 2024)')].index, inplace=True)

plot_data.sort_values(by=['Altitude', 'Category', 'Max to Median Ratio'], ascending=[True, True, False], inplace=True)

sns.barplot(x='Satellite', y='Max to Median Ratio', hue='Category', data=plot_data, ax=ax1)
plt.legend(loc='upper left')
plt.xticks(df_stats_all.index)
plt.ylabel('Max to Median Ratio')
plt.title(f"Max-to-median ratio of orbital decay for all satellites (May / October 2024 highlighted)")

ax2 = plt.twinx()
sns.lineplot(x=df_stats_all.index, y=altitudes, marker='o', label='Altitude (km)', ax=ax2, color='red', linestyle='-', linewidth=5)
ax2.set_ylabel('Altitude (km)')
plt.legend(loc='upper left', bbox_to_anchor=(0, 0.65))
# set x labels to be satellite names from config[satellite]['name']
ax1.set_xticklabels([config[sat]['name'].upper() for sat in df_stats_all.index], rotation=45, ha='right')
# set y axis limit for ax1 to

# plt.tight_layout()
plt.savefig(f"./data/stats_barplot_max_to_median_ratio_{file_label}_all.png")


# Orbit tracks

In [147]:
# check largest decay events in 2023-2024
df_base = dfs['SWMA_RDAWFI'].copy()

disturbed_decay = df_base[df_base['orbital_decay_rate'] > df_base['orbital_decay_rate'].mean() + 5 * df_base['orbital_decay_rate'].std()]
dates = np.unique(disturbed_decay.index.date)

print("Dates of significant decay events:")
target_times = []
for date in dates:
    start = pd.to_datetime(date) - pd.Timedelta(days=1)
    duration = pd.Timedelta(days=3)
    target_times.append((start, duration))
    print(f"  Start: {start}, Duration: {duration}")

# merge overlapping/consecutive windows
merged = []
for start, duration in target_times:
    end = start + duration
    if merged and start <= merged[-1][1]:  # if overlaps with last window
        # extend the last window's end if needed
        merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        print(f"  Merging with previous window, new end: {merged[-1][1]}")
    else:
        merged.append((start, end))

# convert to expected format for plot_orbit_tracks and filter out windows longer than 6 days for compute
target_times = [(start, duration) for start, end in merged for duration in [end - start] if (end - start) <= pd.Timedelta(days=6)]

Dates of significant decay events:
  Start: 2024-05-10 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-08-11 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-09 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-10 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-25 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-26 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-27 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-28 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-29 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-30 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-10-31 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-11-01 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-11-02 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-11-03 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-11-04 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-11-05 00:00:00, Duration: 3 days 00:00:00
  Start: 2024-11-06 00:00:00, Duration: 3 days 00:00:00
  Start: 2024

In [149]:
print("Strongest decay event windows:")
for start, duration in target_times:
    print(f"  Start: {start}, Duration: {duration}")
    end = start + duration
    print(f"Max decay: {df_base[(df_base.index >= start) & (df_base.index <= end)]['orbital_decay_rate'].max(), df_base[(df_base.index >= start) & (df_base.index <= end)]['orbital_decay_rate'].idxmax()}")

Strongest decay event windows:
  Start: 2024-05-10 00:00:00, Duration: 3 days 00:00:00
Max decay: (288.6205, Timestamp('2024-05-11 03:08:00'))
  Start: 2024-08-11 00:00:00, Duration: 3 days 00:00:00
Max decay: (224.937, Timestamp('2024-08-12 08:55:20'))
  Start: 2024-10-09 00:00:00, Duration: 4 days 00:00:00
Max decay: (290.904, Timestamp('2024-10-11 01:57:40'))


In [150]:
for target_time, duration in target_times:
    print(f"\nPlotting orbit tracks for window starting at {target_time} with duration {duration}")
    plot_orbit_tracks(dfs, str(target_time), duration, '1min', config, ['F10.7', 'Hp30', 'SymH'])


Plotting orbit tracks for window starting at 2024-05-10 00:00:00 with duration 3 days 00:00:00
Generating orbit tracks plot for time: 2024-05-10 00:00:00 with duration: 3 days 00:00:00 and resolution: 1min

Plotting orbit tracks for window starting at 2024-08-11 00:00:00 with duration 3 days 00:00:00
Generating orbit tracks plot for time: 2024-08-11 00:00:00 with duration: 3 days 00:00:00 and resolution: 1min

Plotting orbit tracks for window starting at 2024-10-09 00:00:00 with duration 4 days 00:00:00
Generating orbit tracks plot for time: 2024-10-09 00:00:00 with duration: 4 days 00:00:00 and resolution: 1min
